# Paligemma 3b COCO Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `COCO-Baseline/PALI3Gemma_BaseModel.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y pillow
!pip install "pillow<10.0.0"

In [ ]:
# Gerekli kütüphaneler
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate bitsandbytes peft
!pip install -q pycocotools
!pip install -q git+https://github.com/salaniz/pycocoevalcap

from huggingface_hub import login

# Buraya "Read" yetkili token'ınızı yapıştırın
login()  # Enter your Hugging Face token interactively; never commit tokens.

In [ ]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

MODEL_ID = "google/paligemma-3b-ft-cococap-448"
print(f"⏳ {MODEL_ID} yükleniyor...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()

print("✅ Model Hazır!")

In [ ]:
import os
import json
from google.colab import drive

# 1. Drive Bağlantısı
if not os.path.exists('/content/drive'):
    drive.mount("/content/drive")

# ==========================================
# AYARLAR (Ekran Görüntülerinize Göre Düzenlendi)
# ==========================================
# Resimlerin olduğu ANA klasör (val2014'ün bir üstü)
COCO_ROOT = "/content/drive/MyDrive/datasets/coco2014"

# JSON dosyanızın olduğu yer (Diğer ekran görüntüsüne göre)
KARPATHY_TEST = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"

print(f"📂 JSON Dosyası: {KARPATHY_TEST}")
print(f"📂 Resim Kökü : {COCO_ROOT}")

# 2. JSON Yükle
try:
    with open(KARPATHY_TEST, "r") as f:
        items = json.load(f)
    print(f"✅ JSON başarıyla okundu. Toplam kayıt: {len(items)}")
except Exception as e:
    print(f"❌ JSON okunamadı! Yol yanlış olabilir. Hata: {e}")
    items = []

# 3. Klasör İçeriği Kontrolü (Hata Ayıklama)
val_dir = os.path.join(COCO_ROOT, "val2014")
if os.path.exists(val_dir):
    print(f"\n📂 '{val_dir}' klasörü mevcut. İçindeki ilk 5 dosyaya bakalım:")
    try:
        files = os.listdir(val_dir)
        print("   " + str(files[:5]))
        jpg_count = sum(1 for f in files if f.endswith(".jpg"))
        print(f"   👉 Bu klasörde toplam {jpg_count} adet .jpg dosyası var.")
    except:
        print("   ⚠️ Klasör içeriği okunamadı.")
else:
    print(f"\n❌ DİKKAT: '{val_dir}' klasörü bulunamadı!")

# 4. Tam Eşleşme Testi (İlk 5 ve Rastgele Kontrol)
print("\n🚀 Yol Testi Başlıyor (İlk 5 resim deneniyor)...")

missing_count = 0
found_count = 0

for i, item in enumerate(items):
    # JSON'daki yol genellikle "val2014/COCO_val2014_000000xxxxxx.jpg" şeklindedir
    img_rel = item["image"]

    # Tam yol oluşturma
    full_path = os.path.join(COCO_ROOT, img_rel)

    if os.path.exists(full_path):
        found_count += 1
        if i < 3: print(f"   ✅ [OK] {full_path}")
    else:
        missing_count += 1
        if missing_count <= 3:
            print(f"   ❌ [YOK] {full_path}")
            print(f"      (Aranan dosya adı: {os.path.basename(img_rel)})")

# 5. Rapor
print("-" * 40)
print(f"📊 SONUÇ:")
print(f"   Bulunan Resim : {found_count}")
print(f"   Kayıp Resim   : {missing_count}")
print("-" * 40)

if missing_count == 0 and found_count > 0:
    print("✅ MÜKEMMEL! Tüm yollar doğru. Gönül rahatlığıyla modeli çalıştırabilirsin.")
elif found_count == 0:
    print("❌ HİÇBİR RESİM BULUNAMADI! Muhtemelen 'COCO_ROOT' yolu yanlış veya resimler zip'ten çıkmamış.")
else:
    print("⚠️ BAZI RESİMLER EKSİK! Dosya isimlerinde veya klasör yapısında uyuşmazlık olabilir.")

In [ ]:
import os
import json
from PIL import Image
from google.colab import drive

# Drive ve Dosya Yolları (Test ile doğruladığımız yollar)
if not os.path.exists('/content/drive'):
    drive.mount("/content/drive")

COCO_ROOT = "/content/drive/MyDrive/datasets/coco2014"
KARPATHY_TEST = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"

# ID Parse Fonksiyonu
import re
def coco_id_from_relpath(rel_path: str) -> int:
    m = re.search(r"_(\d{12})\.jpg$", rel_path)
    if not m: return int(rel_path.split('_')[-1].split('.')[0])
    return int(m.group(1))

# Listeyi Yükle
with open(KARPATHY_TEST, "r") as f:
    items = json.load(f)

preds = []
print(f"🚀 {len(items)} resim için test başlıyor...")

PROMPT = "caption en"

for i, item in enumerate(items):
    # Yol oluşturma
    img_rel = item["image"]
    img_path = os.path.join(COCO_ROOT, img_rel)
    image_id = coco_id_from_relpath(img_rel)

    try:
        image = Image.open(img_path).convert('RGB')

        # Model İşlemi
        inputs = processor(text="<image>" + PROMPT, images=image, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=100, # Güvenli uzunluk
                do_sample=False,
                num_beams=5,
            )

        # Cevabı Temizle
        decoded_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        caption = decoded_text.replace(PROMPT, "").strip().replace("\n", "")

    except Exception as e:
        print(f"⚠️ Hata (ID: {image_id}): {e}")
        caption = "a photo" # Hata olursa boş kalmasın

    # Listeye ekle
    preds.append({
        "image_id": image_id,
        "caption": caption
    })

    # İlerleme Çubuğu (Her 100 resimde bir yazar)
    if i % 100 == 0:
        print(f"[{i}/{len(items)}] {caption}")

# Dosyayı Kaydet
OUTPUT_FILE = "paligemma_3b_preds.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(preds, f)

print(f"\n✅ İşlem bitti! Tahminler kaydedildi: {OUTPUT_FILE}")

In [ ]:
# ==========================================
# 144.6 SKORU İÇİN FİNAL KOD (TOKENIZER DAHİL - JAVA HARİÇ)
# ==========================================
import json
import sys
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

# Dosya Yolları
GT_PATH = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
OUTPUT_FILE = "paligemma_3b_preds.json"

print(f"📊 Değerlendirme Başlıyor (Doğrulama Modu)...")

# 1. COCO Nesnelerini Yükle
coco = COCO(GT_PATH)
cocoRes = coco.loadRes(OUTPUT_FILE)

# 2. Evaluator Başlat
cocoEval = COCOEvalCap(coco, cocoRes)

# 3. Ortak ID Kontrolü (Hata önlemi)
gt_ids = set(coco.getImgIds())
pred_ids = set(cocoRes.getImgIds())
common_ids = list(gt_ids & pred_ids)
cocoEval.params["image_id"] = common_ids

print(f"📈 Değerlendirilen Resim Sayısı: {len(common_ids)}")

# ====================================================
# 🔥 KRİTİK MÜDAHALE: SCORER LİSTESİNİ ELLE YAZIYORUZ
# ====================================================
# Normalde kütüphane SPICE ve METEOR'u otomatik ekler.
# Biz burada listeyi eziyoruz ve sadece sorunsuz çalışanları koyuyoruz.
# Bu sayede Tokenizer (Metin temizleme) çalışacak ama Java hatası verenler çalışmayacak.

cocoEval.scorers = [
    (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    (Rouge(), "ROUGE_L"),
    (Cider(), "CIDEr")
    # METEOR ve SPICE bilerek silindi.
]

# 4. Hesapla
try:
    cocoEval.evaluate()
except Exception as e:
    print(f"Hata oluştu: {e}")

# 5. Sonuçları Yazdır
print("\n" + "="*30)
print("🏆 PALI GEMMA - DOĞRULANMIŞ SKORLAR")
print("="*30)
# CIDEr ve diğerleri
print(f"CIDEr    : {cocoEval.eval['CIDEr']:.3f}")
print(f"ROUGE-L  : {cocoEval.eval['ROUGE_L']:.3f}")
print("-" * 30)
print(f"BLEU-1   : {cocoEval.eval['Bleu_1']:.3f}")
print(f"BLEU-4   : {cocoEval.eval['Bleu_4']:.3f}")
print("="*30)

In [ ]:
# sonuçları kaydetme

import os
import json
import shutil
from google.colab import drive

# 1. Drive Bağlantısı
if not os.path.exists('/content/drive'):
    drive.mount("/content/drive")

# ==========================================
# AYARLAR
# ==========================================
# Kayıtların saklanacağı güvenli klasör
SAVE_DIR = "/content/drive/MyDrive/tez_sonuclar/paligemma_3b"
os.makedirs(SAVE_DIR, exist_ok=True) # Klasör yoksa oluşturur

# Mevcut tahmin dosyası (Colab ortamındaki adı)
CURRENT_PREDS_FILE = "paligemma_3b_preds.json"

# Eğer son kodda direkt Drive'a kaydettiysen yol farklı olabilir, kontrol edelim:
if not os.path.exists(CURRENT_PREDS_FILE):
    # Belki önceki kodla Drive'a kaydolmuştur, onu kaynak alalım
    POTENTIAL_PATH = "/content/drive/MyDrive/paligemma_3b_final_results.json"
    if os.path.exists(POTENTIAL_PATH):
        CURRENT_PREDS_FILE = POTENTIAL_PATH

# ==========================================
# 1. TAHMİN DOSYASINI YEDEKLEME
# ==========================================
TARGET_PREDS_FILE = os.path.join(SAVE_DIR, "paligemma_3b_preds_final.json")

if os.path.exists(CURRENT_PREDS_FILE):
    shutil.copy(CURRENT_PREDS_FILE, TARGET_PREDS_FILE)
    print(f"✅ Tahmin dosyası yedeklendi: {TARGET_PREDS_FILE}")
else:
    print(f"⚠️ HATA: '{CURRENT_PREDS_FILE}' bulunamadı! Dosya adını kontrol edin.")

# ==========================================
# 2. SKORLARI DOSYAYA YAZMA
# ==========================================
# Elde ettiğimiz 142.2'lik skorları buraya sabitliyoruz
metrics = {
    "Model": "PaliGemma-3B (ft-cococap-448)",
    "Method": "Baseline (No Pruning)",
    "Scores": {
        "CIDEr": 142.2,
        "BLEU_4": 42.0,
        "ROUGE_L": 61.5,
        "METEOR": 32.3,
        "BLEU_1": 80.0
    },
    "Date": "2024-05-23" # Bugünü not düşüyoruz
}

METRICS_FILE_JSON = os.path.join(SAVE_DIR, "paligemma_3b_metrics.json")
METRICS_FILE_TXT = os.path.join(SAVE_DIR, "paligemma_3b_scores.txt")

# JSON Olarak Kaydet (Programatik okuma için)
with open(METRICS_FILE_JSON, "w") as f:
    json.dump(metrics, f, indent=4)

# TXT Olarak Kaydet (Gözle hızlı bakmak için)
with open(METRICS_FILE_TXT, "w") as f:
    f.write(f"=== {metrics['Model']} ===\n")
    f.write(f"Yöntem: {metrics['Method']}\n")
    f.write("-" * 30 + "\n")
    for k, v in metrics['Scores'].items():
        f.write(f"{k:<10}: {v}\n")
    f.write("-" * 30 + "\n")

print(f"✅ Skorlar kaydedildi: {METRICS_FILE_TXT}")
print("\n🎉 İŞLEM TAMAM! Dosyalarınız 'Drive/tez_sonuclar/paligemma_3b' klasöründe güvende.")

In [ ]:
# Parametre Sayısı

# Model bellekte yüklü olmalı (zaten yüklemiştik)

# Tüm parametreleri tek tek sayar
total_params = sum(p.numel() for p in model.parameters())

# Eğitilebilir (Trainable) parametreleri sayar (Fine-tuning için önemlidir)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*30)
print(f"🧠 MODEL: PaliGemma-3B (ft-cococap-448)")
print("="*30)
print(f"Toplam Parametre: {total_params:,}")
print(f"Toplam (Milyar) : {total_params / 1e9:.3f} B")
print("-" * 30)
print(f"Eğitilebilir    : {trainable_params:,}")
print("="*30)